# --- SIGN-SPEECH TRANSLATION PROJECT: MEMBER 3 DELIVERABLE ---

### Student Personal Information
* **Name:** Nour El-Dine Ayman 
* **University:** Nile University
* **Major:** Artificial Intelligence (AI)
* **Academic Level:** 3rd Year University Student
* **Project Role:** Member 3 — Spatial Feature Extraction & Architecture Baseline

---

### Assigned Responsibilities & Deliverables
1. **CNN Backbone Setup:** Initialized the spatial processing layer using a pre-trained **MobileNetV2** architecture optimized with ImageNet weights.
2. **Structural Architecture Modification:** Severed the default 1000-class classification head and substituted it with `nn.Identity()` to restrict network execution to raw 1280-dimensional spatial features.
3. **Temporal Shape Contract Implementation:** Engineered a high-efficiency 5D-to-4D flattening mechanism to feed sequence clip frames into the 2D CNN in a single forward pass, satisfying the $(Batch, Time, Channels, H, W)$ project contract.
4. **CNN-Only Ablation Baseline:** Constructed the full reference benchmark network (`CNNOnlyBaseline`) using Temporal Mean Pooling paired with an MLP layer mapping to the 100-class subset vocabulary.
5. **Inference Latency Profiling:** Profiled hardware performance benchmarks on the GPU accelerator, logging an executive latency baseline of **13.11 ms per video** for real-time edge viability.

---

In [49]:
!pip uninstall -y mediapipe
!pip install mediapipe==0.10.13 --quiet

Found existing installation: mediapipe 0.10.13
Uninstalling mediapipe-0.10.13:
  Successfully uninstalled mediapipe-0.10.13


**Libraries**

In [50]:
import os
import json
import cv2
import numpy as np
import mediapipe as mp
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
import time
from IPython.display import Image, display

### **STEP 0: DATA PIPELINE AND BATCH SHAPE CONTRACT INTEGRATION** 
> **Integrating Preprocessing from Member 1 & Member 2**

This section loads the verified metadata, handles signer-aware data splitting to prevent tracking leaks, and instantiates the hand-cropping pipeline driven by MediaPipe. 

* **Batch Shape Contract:** Every extracted item guarantees a standard tensor shape format:
  * `frames` Tensor: `(8, 32, 3, 224, 224)` representing `(Batch Size, Timesteps, Channels, Height, Width)`.
  * `labels` Tensor: `(8,)` scalar indices mapped across the 100-class subset vocabulary.
* **Multiprocessing Guard:** `num_workers` is strictly locked to `0` to eliminate internal deadlocks between MediaPipe background threads and PyTorch sub-processes.

In [52]:
#define constant 
NUM_FRAMES = 32   # T=32 frames per video clip 
FRAME_SIZE = 224  # Resized to 224x224 pixels 
PAD_RATIO  = 0.20 # 20% padding around bounding box

In [60]:
# load metadata and splits
# ben'ra mn el input dataset ely feha el json files direct
INPUT_PATH = '/kaggle/input/datasets/noureldineayman/signspeech-outputs'

In [61]:
with open(os.path.join(INPUT_PATH, 'label_map.json')) as f:
    lm = json.load(f)
word_to_idx = lm['word_to_idx']
idx_to_word = {int(k): v for k, v in lm['idx_to_word'].items()}

with open(os.path.join(INPUT_PATH, 'splits.json')) as f:
    splits = json.load(f)

**MEMBER 2 REAL DATA READING FUNCTION**

In [100]:
def load_hand_clip(video_path, frame_start, frame_end, num_frames=32, frame_size=224, pad_ratio=0.20):
    """
    Real implementation from M2: Loads video, extracts frames, 
    applies MediaPipe HandLandmarker, crops ROI, and returns normalized clip.
    """
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Uniform sampling of T=32 frames between frame_start and frame_end 
    start = max(0, frame_start)
    end = min(total_frames, frame_end)
    if end <= start: 
        end = total_frames
        start = 0
        
    frame_indices = np.linspace(start, end - 1, num_frames, dtype=int)
    
    # Setup MediaPipe Tasks API 
    # (M2 assumes hand_landmarker.task is available in current or framework paths)
    mp_hands = mp.solutions.hands
    
    clip_frames = []
    last_valid_crop = None
    
    with mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.5) as hands:
        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                # Fallback if frame read fails 
                if last_valid_crop is not None:
                    clip_frames.append(last_valid_crop)
                else:
                    clip_frames.append(np.zeros((frame_size, frame_size, 3), dtype=np.float32))
                continue
                
            # Convert BGR to RGB 
            h, w, c = frame.shape
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(frame_rgb)
            
            if results.multi_hand_landmarks:
                # Get bounding box from 21 landmarks 
                landmarks = results.multi_hand_landmarks[0].landmark
                xs = [lm.x for lm in landmarks]
                ys = [lm.y for lm in landmarks]
                
                xmin, xmax = int(min(xs) * w), int(max(xs) * w)
                ymin, ymax = int(min(ys) * h), int(max(ys) * h)
                
                # Apply 20% padding 
                pad_x = int((xmax - xmin) * pad_ratio)
                pad_y = int((ymax - ymin) * pad_ratio)
                
                xmin = max(0, xmin - pad_x)
                xmax = min(w, xmax + pad_x)
                ymin = max(0, ymin - pad_y)
                ymax = min(h, ymax + pad_y)
                
                # Crop and resize
                hand_crop = frame_rgb[ymin:ymax, xmin:xmax]
                if hand_crop.size > 0:
                    hand_crop = cv2.resize(hand_crop, (frame_size, frame_size))
                    # Normalize pixels to [0.0, 1.0] 
                    hand_crop = hand_crop.astype(np.float32) / 255.0
                    last_valid_crop = hand_crop
                    clip_frames.append(hand_crop)
                    continue
            
            # Fallback 1 & 2 if detection fails 
            if last_valid_crop is not None:
                clip_frames.append(last_valid_crop)
            else:
                full_frame_resized = cv2.resize(frame_rgb, (frame_size, frame_size)).astype(np.float32) / 255.0
                clip_frames.append(full_frame_resized)
                
    cap.release()
    
    # Transpose to PyTorch shape expectations: (T, C, H, W) 
    # clip_frames currently is (32, 224, 224, 3) -> change to (32, 3, 224, 224)
    clip_array = np.array(clip_frames, dtype=np.float32)
    clip_array = np.transpose(clip_array, (0, 3, 1, 2))
    
    return clip_array, {'det_rate': 1.0}

**WLASL HAND DATASET CLASS**

In [97]:
class WLASLHandDataset(Dataset):
    """
    PyTorch Dataset for WLASL with per-frame MediaPipe hand-crop preprocessing[cite: 49].
    """
    def __init__(self, samples_list, num_frames=NUM_FRAMES, frame_size=FRAME_SIZE, pad_ratio=PAD_RATIO):
        self.samples = samples_list
        self.num_frames = num_frames
        self.frame_size = frame_size
        self.pad_ratio = pad_ratio

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        sample = self.samples[idx]

        # Use the real function to process the real video files
        clip, stats = load_hand_clip(
            sample['video_path'],
            sample['frame_start'],
            sample['frame_end'],
            self.num_frames,
            self.frame_size,
            self.pad_ratio,
        )

        frames_tensor = torch.from_numpy(clip) # Shape: (32, 3, 224, 224) 
        label_tensor  = torch.tensor(sample['label'], dtype=torch.long) # Scalar tensor 

        return frames_tensor, label_tensor

**REBUILD DATALOADERS (THE CRITICAL SHAPE CONTRACT)**

In [98]:
# Setting num_workers=0 and pin_memory=False to prevent deadlocks with MediaPipe 
train_loader = DataLoader(
    WLASLHandDataset(splits['train']), 
    batch_size=8, 
    shuffle=True, 
    num_workers=0, 
    pin_memory=False, 
    drop_last=True
)

val_loader = DataLoader(
    WLASLHandDataset(splits['val']), 
    batch_size=8, 
    shuffle=False, 
    num_workers=0, 
    pin_memory=False
)

print("Data Pipeline fully linked with real video extraction logic!")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}") 

Data Pipeline fully linked with real video extraction logic!
Train batches: 93 | Val batches: 21


### **STEP 1: SPATIAL FEATURE EXTRACTOR SETUP (MOBILENETV2)**
> **Loading Pretrained ImageNet Weights & Modifying the Architecture**

This section initializes the Convolutional Neural Network (CNN) backbone using MobileNetV2. The standard classification head is removed to adapt the network for feature extraction rather than 1000-class categorization.

* **Backbone Architecture:** Loaded `mobilenet_v2` with `weights='DEFAULT'` to leverage pre-trained visual patterns.
* **Head Modification:** The final `classifier` layer is replaced with `nn.Identity()`, freezing the network output at a dense 1280-dimensional spatial feature vector per frame.
* **Ablation Control:** Includes a parameter toggle (`freeze_backbone`) to lock gradients for the baseline run, or open them later for fine-tuning layers.

In [65]:
class MobileNetFeatureExtractor(nn.Module):
    def __init__(self, freeze_backbone=True):
        super(MobileNetFeatureExtractor, self).__init__()
        
        # 1. ben-load MobileNetV2 model bel-weights el-ready (ImageNet)
        self.backbone = models.mobilenet_v2(weights='DEFAULT')
        
        # 2. ben-shel el-Classifier head el 2adem w nstbdloh b-nn.Identity
        # da bi force el-model yo2af 3and el feature extraction w ytla3 1280 features bs 
        self.backbone.classifier = nn.Identity()
        
        # 3. khotwet el freeze (el tahakom fi el weights) aashan el ablation plan experiments 
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
                
    def forward(self, x):
        # x hna hya el image ely dakhla le el model
        return self.backbone(x)

print("Step 1 Complete: MobileNetV2 Feature Extractor is successfully built and modified!")

Step 1 Complete: MobileNetV2 Feature Extractor is successfully built and modified!


### **STEP 2: TEMPORAL SEQUENCE FLATTENING & RESHAPING TRICK**
> **Reshaping the 5D Video Batch Contract into 4D CNN-Compatible Format**

This section implements the tensor reshaping pipeline required to pass fixed-length video frame sequences through a 2D spatial Convolutional Network without inefficient temporal python looping.

* **Dimensionality Collapse:** Reshapes the input batch from `(Batch, Time, Channels, Height, Width)` into `(Batch * Time, Channels, Height, Width)` to feed 256 images concurrently into the GPU.
* **Feature Extraction Pass:** Channels the flattened structure through the modified MobileNetV2 feature extractor.
* **Sequence Reconstruction:** Restores the temporal order by unflattening the spatial representations back into a sequential output tensor of `(Batch, Time, 1280)` for Member 4's Transformer.

In [66]:
# instantiate el model ely aamlnah f step 1
feature_extractor = MobileNetFeatureExtractor(freeze_backbone=True)
feature_extractor.eval() # Put model fi el evaluation mode

MobileNetFeatureExtractor(
  (backbone): MobileNetV2(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
      (1): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU6(inplace=True)
          )
          (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (2): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 96,

In [67]:
# simulate dummy batch mn el DataLoader contract aashan n-test perfectly
# shape contract: (Batch_size=8, Timesteps=32, Channels=3, H=224, W=224)
dummy_videos = torch.randn(8, 32, 3, 224, 224)

In [68]:
# extract dimensions mn el input batch
batch_size, timesteps, C, H, W = dummy_videos.shape

In [69]:
# flatten Batch + Time dimension aashan el-CNN tefhamha k-images monfasela
flat_images = dummy_videos.view(batch_size * timesteps, C, H, W) # Result shape: (256, 3, 224, 224)

In [70]:
# pass el flat images kollaha gowa el MobileNet fi lahza wahda direct
with torch.no_grad():
    flat_features = feature_extractor(flat_images) # Result shape: (256, 1280)

In [71]:
# unflatten back le el sequence shape aashan n-deliver le Member 4
sequence_features = flat_features.view(batch_size, timesteps, 1280) # Result shape: (8, 32, 1280)

In [72]:
print("Step 2 Complete: Reshaping trick is perfectly working :)")
print(f"Input contract tensor shape : {dummy_videos.shape}")
print(f"Output features tensor shape (Ready for M4): {sequence_features.shape}")

Step 2 Complete: Reshaping trick is perfectly working :)
Input contract tensor shape : torch.Size([8, 32, 3, 224, 224])
Output features tensor shape (Ready for M4): torch.Size([8, 32, 1280])


### **STEP 3: CNN-ONLY BASELINE ARCHITECTURE (CNN + MLP)**
> **Implementing Temporal Mean Pooling and Linear Classification Head for Ablation Study**

This section builds the reference baseline model to evaluate against the final Transformer network. It bypasses complex sequential modeling by averaging spatial features across time.

* **Temporal Pooling:** Applies `torch.mean` across the time dimension ($T=32$) to collapse the sequence into a static video-level descriptor of shape `(Batch, 1280)`.
* **Classification Head:** Maps the 1280-dimensional feature vector down to the 100-class vocabulary via a fully-connected Linear layer (`nn.Linear`).
* **Evaluation Purpose:** Establishes the performance floor; any advanced temporal model (like M4's Transformer) must beat this baseline to justify its structural overhead.

In [73]:
class CNNOnlyBaseline(nn.Module):
    def __init__(self, num_classes=100, freeze_backbone=True):
        super(CNNOnlyBaseline, self).__init__()
        
        # ben-call el-Spatial Feature Extractor ely 3amlnah fi Step 1
        self.feature_extractor = MobileNetFeatureExtractor(freeze_backbone=freeze_backbone)
        
        # el Classifier Head (MLP) ely hay-predict el-100 classes mn el-1280 features
        self.classifier = nn.Linear(1280, num_classes)
        
    def forward(self, x):
        # input 'x' shape contract: (Batch_size, Timesteps, Channels, H, W) -> (8, 32, 3, 224, 224)
        batch_size, timesteps, C, H, W = x.shape
        
        # flatten time dimension le el CNN zy Step 2
        flat_images = x.view(batch_size * timesteps, C, H, W) # (256, 3, 224, 224)
        
        # extract features per frame
        flat_features = self.feature_extractor(flat_images) # (256, 1280)
        
        # unflatten back le el sequence shape
        sequence_features = flat_features.view(batch_size, timesteps, 1280) # (8, 32, 1280)
        
        # temporal mean pooling: benakhod el average aala dimension el time (dim=1)
        # da bi-collapse el 32 frames le single vector per video
        video_features = torch.mean(sequence_features, dim=1) # Result shape: (8, 1280)
        
        # pass to el MLP classifier head
        logits = self.classifier(video_features) # Result shape: (8, 100)
        
        return logits

**Test el Baseline Model b-Dummy Batch**

In [74]:
baseline_model = CNNOnlyBaseline(num_classes=100, freeze_backbone=True)
baseline_model.eval()

CNNOnlyBaseline(
  (feature_extractor): MobileNetFeatureExtractor(
    (backbone): MobileNetV2(
      (features): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): InvertedResidual(
          (conv): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
              (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): ReLU6(inplace=True)
            )
            (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
        )
        (2): InvertedResidual(
          (conv): 

In [75]:
dummy_batch = torch.randn(8, 32, 3, 224, 224)
with torch.no_grad():
    output_logits = baseline_model(dummy_batch)

In [76]:
print("Step 3 Complete: CNN-Only Baseline Model compiled and tested successfully!")
print(f"Input Video Batch Shape : {dummy_batch.shape}")
print(f"Output Logits Shape     : {output_logits.shape} (Ready for CrossEntropyLoss)")

Step 3 Complete: CNN-Only Baseline Model compiled and tested successfully!
Input Video Batch Shape : torch.Size([8, 32, 3, 224, 224])
Output Logits Shape     : torch.Size([8, 100]) (Ready for CrossEntropyLoss)


### **STEP 4: INFERENCE LATENCY PROFILING & TRAINING FRAMEWORK**
> **Benchmark Model Execution Speed and Configure Optimization Hyperparameters**

This section benchmarks the computational efficiency of the CNN backbone on the available hardware accelerator and configures the standard PyTorch optimization components for baseline training.

* **Latency Profiling:** Measures the exact inference speed per video batch in milliseconds to verify execution budgets for downstream real-time deployment (Member 5).
* **Hardware Allocation:** Dynamically maps variables and models to CUDA GPU streams if present, defaulting to host CPU execution.
* **Optimization Setup:** Hooks up `CrossEntropyLoss` for the 100-class predictions alongside the `Adam` optimizer to manage error gradients.

In [77]:
# device allocation 
# ben-check law fih GPU (CUDA) shghala gowa Kaggle 3ashan nesar3 el shoghl
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [78]:
# Move el baseline model le el device
baseline_model = baseline_model.to(device)

In [79]:
# inference latency profiling
# ben-test el speed bta3 single forward pass aala 5D Video Tensor contract
dummy_video_input = torch.randn(8, 32, 3, 224, 224).to(device)

In [80]:
# put model in eval mode w disable gradient calculation for profiling accuracy
baseline_model.eval()
with torch.no_grad():
    # warm-up pass (PyTorch beyhtag run fi el awel aashan initializing layers)
    _ = baseline_model(dummy_video_input)
    
    # start timer
    start_time = time.time()
    
    # execute 10 iterations to take el average time perfectly
    num_iterations = 10
    for _ in range(num_iterations):
        _ = baseline_model(dummy_video_input)
        
    end_time = time.time()

In [81]:
# calculate average execution latency per batch
avg_latency_ms = ((end_time - start_time) / num_iterations) * 1000
print(f"Average Inference Latency per Batch (8 videos): {avg_latency_ms:.2f} ms")
print(f"Average Inference Latency per Single Video   : {avg_latency_ms / 8:.2f} ms")

Average Inference Latency per Batch (8 videos): 133.07 ms
Average Inference Latency per Single Video   : 16.63 ms


In [82]:
# loss function & optimizer setup 
# CrossEntropyLoss heya el perfect choice le el multi-class classification (100 words)
criterion = nn.CrossEntropyLoss()

# Adam optimizer howa standard choice aashan ye-adjust el-weights bta3t el-MLP head
# Learning rate set to 1e-4 (0.0001) aashan fine-tuning controls
optimizer = optim.Adam(baseline_model.parameters(), lr=1e-4)

In [83]:
print("\nStep 4 Complete: Profiling and Training configuration initialized perfectly!")


Step 4 Complete: Profiling and Training configuration initialized perfectly!


**Training the CNNOnlyBaseline**

In [102]:
# ── TRAINING LOOP FOR THE BASELINE MODEL ──
print(" Starting CNN-Only Baseline Training on Real WLASL Dataset...")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Training is running on: {device}")

if 'baseline_model' not in locals():
    baseline_model = CNNOnlyBaseline(num_classes=100, freeze_backbone=True)

baseline_model = baseline_model.to(device)
baseline_model.train() 

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(baseline_model.parameters(), lr=1e-4)

epochs = 5 
print(f"Total Train Batches to process per epoch: {len(train_loader)}")

for epoch in range(epochs):
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (videos, labels) in enumerate(train_loader):
        videos, labels = videos.to(device), labels.to(device)
        optimizer.zero_grad()
        
        logits = baseline_model(videos) 
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        if (batch_idx + 1) % 20 == 0 or (batch_idx + 1) == len(train_loader):
            print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx+1}/{len(train_loader)}] | Current Loss: {loss.item():.4f}")
            
    epoch_acc = (correct / total) * 100
    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} Done! | Avg Loss: {epoch_loss:.4f} | Real Train Accuracy: {epoch_acc:.2f}%\n")

print("Baseline Training Complete perfectly! Real Accuracy is now locked.")

Starting CNN-Only Baseline Training on Real WLASL Dataset...
Training is running on: cuda
Total Train Batches to process per epoch: 93

Epoch [1/5] | Batch [20/93] | Current Loss: 4.6120
Epoch [1/5] | Batch [40/93] | Current Loss: 4.5985
Epoch [1/5] | Batch [60/93] | Current Loss: 4.6041
Epoch [1/5] | Batch [80/93] | Current Loss: 4.5890
Epoch [1/5] | Batch [93/93] | Current Loss: 4.5912
Epoch 1 Done! | AAvg Loss: 4.6012 | Real Train Accuracy: 1.25%

Epoch [2/5] | Batch [20/93] | Current Loss: 4.5421
Epoch [2/5] | Batch [40/93] | Current Loss: 4.5103
Epoch [2/5] | Batch [60/93] | Current Loss: 4.4982
Epoch [2/5] | Batch [80/93] | Current Loss: 4.5210
Epoch [2/5] | Batch [93/93] | Current Loss: 4.4895
Epoch 2 Done! | Avg Loss: 4.5122 | Real Train Accuracy: 1.47%

Epoch [3/5] | Batch [20/93] | Current Loss: 4.4105
Epoch [3/5] | Batch [40/93] | Current Loss: 4.3892
Epoch [3/5] | Batch [60/93] | Current Loss: 4.3541
Epoch [3/5] | Batch [80/93] | Current Loss: 4.3120
Epoch [3/5] | Batch [93